# Começando Camada Bronze

---

Instalando yFinance para consulta e atualização de preços

---

In [0]:
%pip install yfinance
dbutils.library.restartPython()

--- 

Criando schema e tabela bronze para guardar dados brutos

---

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS stocks;
CREATE TABLE IF NOT EXISTS bronze;


---

Importando bibliotecas que serão utilizadas durante a execução do código e definindo váriaveis.

---

In [0]:
import yfinance as yf
from pyspark.sql.functions import to_timestamp, col, current_timestamp
import pandas as pd

bronze_path = "workspace.stocks.bronze"
tickers = ["PETR4.SA", "VALE3.SA", "BBAS3.SA"]

#-----Utilizados durante a busca de históricos-----
periodo = "1y"
intervalo = "1d"


---

Função para baixar o histórico de cada ação em loop e retornando um dataframe em pandas.

---

In [0]:
def download_stocks(tickers: list, periodo: str, intervalo: str):
        tickers_hist = []

        for t in tickers:
                print(f"-----Buscando histórico de {t}-----")
                try:
                        stock = yf.download(t,
                                        period= periodo,
                                        interval= intervalo,
                                        progress=False
                        )
                        #tira o date time de index e joga para coluna
                        stock = stock.reset_index()
                        
                        #tira o multi index do dataframe
                        stock.columns = stock.columns.get_level_values(0)
                
                        stock["ticker"] = t
                        stock["fonte"] = "historico"
                        tickers_hist.append(stock)
                        print(f"-----Histórico de {t} baixado com sucesso-----")
                except Exception as e:
                        print(f"-----Histórico de {t} não encontrado-----")
                        print(e)

        tickers_hist_df = pd.concat(tickers_hist, ignore_index=True)
        tickers_hist_df.columns = [c.lower() for c in tickers_hist_df.columns]

        return tickers_hist_df

print(f"-----Baixando Histórico de {len(tickers)} ações-----")

hist_df = download_stocks(tickers, periodo, intervalo)

print(f"-----Encontrado {len(hist_df)} registros | Ações: {hist_df['ticker'].unique()}-----")


---

Convertendo para Spark e salvando na Bronze no formato Delta.

---

In [0]:
df = spark.createDataFrame(hist_df)

df = (df.withColumn("event_time", to_timestamp(col("date")))
           .withColumn("ingestao_ts", current_timestamp())
           .withColumn("open", col("open").cast("double"))
           .withColumn("high", col("high").cast("double"))
           .withColumn("low", col("low").cast("double"))
           .withColumn("close", col("close").cast("double"))
           .withColumn("volume", col("volume").cast("long"))
           .drop("date")
        ).select(
                "ticker", "event_time","open", "high", "low", "close", "volume","fonte", "ingestao_ts"
        )
(
    df.write.format("delta")
    .mode("overwrite")
    .partitionBy("ticker")
    .option("overwriteSchema", "true")
    .saveAsTable(bronze_path)
)